# Aplicaciones Prácticas de NLP con Transformers

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA**
* **Nomenclatura Oficial:** Procesamiento de Lenguaje Natural
* **Nombre de Trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura
* **Alumna:** Dominguez Micaela Belen
---

## Objetivo
Implementar y resolver problemas reales de NLP en entornos corporativos utilizando pipelines de Hugging Face para moderación de contenido, extracción de entidades y sistemas de preguntas y respuestas.

## Resultados de aprendizaje
Al final de este notebook vas a poder:
1. Resolver tareas reales de moderación automática de comentarios en español.
2. Implementar extracción de información estructurada (NER) para procesamiento de CVs.
3. Construir un pipeline de Question Answering para responder preguntas sobre políticas empresariales.
4. Diseñar de forma autónoma una solución completa de análisis de reseñas comerciales.



## Terminología clave (Microglosario)

* **✦ Reconocimiento de Entidades Nombradas (NER):** Tarea de NLP que consiste en localizar y clasificar elementos clave dentro del texto en categorías predefinidas (como nombres de personas, organizaciones, ubicaciones, fechas, etc.).
* **✦ Question Answering (QA / Búsqueda de Respuestas):** Tarea que consiste en extraer o generar de manera automática respuestas precisas a preguntas formuladas en lenguaje natural a partir de un contexto o documento dado.
* **✦ Zero-Shot Classification:** Capacidad de un modelo de clasificar un texto en categorías que no ha visto explícitamente durante su entrenamiento, basándose únicamente en descripciones conceptuales o etiquetas semánticas.
* **✦ Pipeline de NLP:** Serie encadenada de procesos de transformación de texto (tokenización, inferencia, decodificación) ejecutados de forma automática por un modelo.



In [3]:
# Instalación de librerías (ejecutar solo una vez)
!pip install transformers torch pandas -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## Ejercicio 1: Moderación de Comentarios en Redes Sociales

### Contexto del problema

Imaginate que trabajás en el equipo de redes sociales de una empresa argentina de delivery de comida. Recibís cientos de comentarios diarios en Instagram y Facebook. Necesitás un sistema que identifique automáticamente los comentarios negativos o críticos para que el equipo de atención al cliente pueda responderlos con prioridad.

### Aplicación real

Este tipo de sistemas se usan en:
- Redes sociales de empresas (PedidosYa, Rappi, Mercado Libre)
- Plataformas de e-commerce
- Servicios de atención al cliente automatizada

### ¿Qué vamos a hacer?

1. Cargar un modelo de análisis de sentimientos en español
2. Analizar comentarios reales de clientes
3. Clasificarlos automáticamente (positivo, negativo, neutral)
4. Identificar cuáles requieren atención urgente

---

### Paso 1: Importar librerías y cargar el modelo

In [4]:
from transformers import pipeline
import pandas as pd

# Cargamos un modelo de análisis de sentimientos específico para español
# Este modelo fue entrenado con datos de redes sociales en español
clasificador = pipeline(
    "text-classification",
    model="finiteautomata/beto-sentiment-analysis"
)

print("Modelo cargado correctamente.")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]


Modelo cargado correctamente.


c:\dominguez-micaela-belen-pln-1c-2026\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Paso 2: Definir comentarios de ejemplo

Acá tenemos comentarios típicos que podría recibir una empresa de delivery:

In [5]:
comentarios = [
    "La comida llegó rapidísimo y estaba caliente. El repartidor re amable, excelente servicio!",
    "Tardaron DOS HORAS para traer una pizza fría. Pésimo, no pido más acá.",
    "Todo bien, llegó en tiempo y forma. Nada para destacar.",
    "Me encanta este servicio, siempre cumplen. Los re banco!",
    "Se equivocaron con mi pedido OTRA VEZ. Ya es la tercera vez que pasa, un desastre total."
]

### Paso 3: Analizar los comentarios

In [6]:
# Procesamos todos los comentarios
resultados = clasificador(comentarios)

# Creamos un DataFrame para visualizar mejor los resultados
df_resultados = pd.DataFrame({
    'Comentario': comentarios,
    'Sentimiento': [r['label'] for r in resultados],
    'Confianza': [round(r['score'], 3) for r in resultados]
})

df_resultados

,Comentario,Sentimiento,Confianza
0,La comida llegó rapidísimo y estaba caliente. ...,POS,0.998
1,Tardaron DOS HORAS para traer una pizza fría. ...,NEG,0.999
2,"Todo bien, llegó en tiempo y forma. Nada para ...",POS,0.997
3,"Me encanta este servicio, siempre cumplen. Los...",POS,0.999
4,Se equivocaron con mi pedido OTRA VEZ. Ya es l...,NEG,0.999


### Paso 4: Identificar comentarios que requieren atención urgente

In [7]:
# Filtramos solo los comentarios negativos
comentarios_urgentes = df_resultados[df_resultados['Sentimiento'] == 'NEG']

print("COMENTARIOS QUE REQUIEREN ATENCIÓN URGENTE:")
print("="*60)
for idx, fila in comentarios_urgentes.iterrows():
    print(f"\n[{idx+1}] {fila['Comentario']}")
    print(f"    Confianza: {fila['Confianza']*100:.1f}%")

COMENTARIOS QUE REQUIEREN ATENCIÓN URGENTE:

[2] Tardaron DOS HORAS para traer una pizza fría. Pésimo, no pido más acá.
    Confianza: 99.9%

[5] Se equivocaron con mi pedido OTRA VEZ. Ya es la tercera vez que pasa, un desastre total.
    Confianza: 99.9%


## Consigna de Lectura e Interpretación

Ahora es tu turno. Realizá las siguientes tareas:

1. **Agregá 3 comentarios nuevos** a la lista (pueden ser inventados o reales)
2. **Ejecutá nuevamente** las celdas de análisis
3. **Respondé:**
   - ¿El modelo clasificó correctamente tus comentarios?
   - ¿Hubo algún comentario que esperabas que clasifique diferente?
   - ¿Qué pasa si escribís un comentario con ironía? (probá con algo como "Genial, 3 horas esperando, lo mejor!")

Escribí tus comentarios nuevos en la siguiente celda:

In [8]:
# TUS COMENTARIOS ACÁ
mis_comentarios = [
    "Llego en 15 minutos, caliente un lujo total. El repartidor re amable",
    "Hace dos horas hice el pedido y aun no llego, no responden los mensajes. Una verguenza",
    "Genial, tres horas esperando, lo mejor!"
]

# Analizamos
mis_resultados = clasificador(mis_comentarios)
0
# Mostramos
df_mis_resultados = pd.DataFrame({
    'Comentario': mis_comentarios,
    'Sentimiento': [r['label'] for r in mis_resultados],
    'Confianza': [round(r['score'], 3) for r in mis_resultados]
})

df_mis_resultados

,Comentario,Sentimiento,Confianza
0,"Llego en 15 minutos, caliente un lujo total. E...",NEU,0.669
1,"Hace dos horas hice el pedido y aun no llego, ...",NEG,0.999
2,"Genial, tres horas esperando, lo mejor!",POS,0.998


Si el primero es un comentario positivo y me lo tomo como neutral.  Y el último debería ser negativo pero  no capturo el sarcasmo.

---

## Ejercicio 2: Extracción Automática de Información de CVs

### Contexto del problema

Trabajás en el área de Recursos Humanos de una consultora argentina. Recibís decenas de CVs por día y necesitás extraer rápidamente información clave: nombres de candidatos, empresas donde trabajaron, universidades, ciudades, y tecnologías que manejan.

### Aplicación real

Este tipo de sistemas se usan en:
- Plataformas de empleo (LinkedIn, Bumeran, Computrabajo)
- Sistemas de ATS (Applicant Tracking Systems)
- Automatización de procesos de RRHH

### ¿Qué vamos a hacer?

1. Cargar un modelo de Reconocimiento de Entidades Nombradas (NER)
2. Procesar un fragmento de CV
3. Extraer automáticamente: personas, organizaciones, ubicaciones
4. Organizar la información de manera estructurada

---

### Paso 1: Cargar el modelo de NER

In [9]:
# Cargamos un modelo de NER específico para español
# Este modelo fue entrenado para reconocer personas, organizaciones y lugares
extractor_ner = pipeline(
    "ner",
    model="mrm8488/bert-spanish-cased-finetuned-ner",
    aggregation_strategy="simple"  # Agrupa tokens de la misma entidad
)

print("Modelo NER cargado correctamente.")

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Modelo NER cargado correctamente.


c:\dominguez-micaela-belen-pln-1c-2026\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Paso 2: Definir un fragmento de CV

Este es un ejemplo típico de cómo un candidato describe su experiencia:

In [10]:
cv_texto = """María Fernández es una desarrolladora full-stack con 5 años de experiencia.
Estudió Ingeniería en Sistemas en la Universidad de Buenos Aires (UBA) y se graduó en 2018.
Trabajó como desarrolladora backend en Mercado Libre durante 3 años, donde lideró la migración
de sistemas legacy a microservicios. Posteriormente se unió a Globant como tech lead,
coordinando equipos distribuidos entre Buenos Aires y Córdoba. Tiene experiencia en Python,
Java, Docker y Kubernetes. Actualmente reside en Palermo, Ciudad de Buenos Aires."""

print("CV a procesar:")
print(cv_texto)

CV a procesar:
María Fernández es una desarrolladora full-stack con 5 años de experiencia.
Estudió Ingeniería en Sistemas en la Universidad de Buenos Aires (UBA) y se graduó en 2018.
Trabajó como desarrolladora backend en Mercado Libre durante 3 años, donde lideró la migración
de sistemas legacy a microservicios. Posteriormente se unió a Globant como tech lead,
coordinando equipos distribuidos entre Buenos Aires y Córdoba. Tiene experiencia en Python,
Java, Docker y Kubernetes. Actualmente reside en Palermo, Ciudad de Buenos Aires.


### Paso 3: Extraer entidades del CV

In [11]:
# Procesamos el texto del CV
entidades = extractor_ner(cv_texto)

# Creamos un DataFrame para visualizar mejor
df_entidades = pd.DataFrame(entidades)

# Mostramos solo las columnas relevantes
df_entidades[['entity_group', 'word', 'score']].round(3)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,entity_group,word,score
0,PER,María Fernández,1.000
1,MISC,Ingeniería en Sistemas,0.998
2,ORG,Universidad de Buenos Aires,0.998
3,ORG,UBA,0.983
4,MISC,Mercado Libre,0.789
5,ORG,Glo,1.000
6,ORG,##bant,0.821
7,LOC,Buenos Aires,1.000
8,LOC,Córdoba,1.000
9,MISC,Py,0.999


Globant y Python se fragmentaron en subwords y cayeron en MISC.Ademas Mercado libre deberia caer como ORG


### Paso 4: Organizar la información por categoría

Ahora vamos a separar las entidades por tipo para que sea más fácil revisarlas:

In [12]:
# Diccionario para organizar por tipo de entidad
info_extraida = {
    'PER': [],  # Personas
    'ORG': [],  # Organizaciones
    'LOC': []   # Ubicaciones
}

# Clasificamos cada entidad
for entidad in entidades:
    tipo = entidad['entity_group']
    palabra = entidad['word']
    if tipo in info_extraida:
        info_extraida[tipo].append(palabra)

# Mostramos la información organizada
print("INFORMACIÓN EXTRAÍDA DEL CV")
print("="*60)
print(f"\nCANDIDATO/A: {', '.join(info_extraida['PER'])}")
print(f"\nEMPRESAS/INSTITUCIONES: {', '.join(info_extraida['ORG'])}")
print(f"\nUBICACIONES: {', '.join(info_extraida['LOC'])}")

INFORMACIÓN EXTRAÍDA DEL CV

CANDIDATO/A: María Fernández

EMPRESAS/INSTITUCIONES: Universidad de Buenos Aires, UBA, Glo, ##bant

UBICACIONES: Buenos Aires, Córdoba, Palermo, Ciudad de Buenos Aires


## Consigna de Lectura e Interpretación

Ahora es tu turno. Realizá las siguientes tareas:

1. **Escribí un fragmento de CV ficticio** (o usá el tuyo si querés) con:
   - Nombre de la persona
   - Al menos 2 empresas o instituciones educativas
   - Al menos 2 ubicaciones (ciudades, barrios, países)

2. **Ejecutá el análisis** y verificá qué entidades detectó el modelo

3. **Respondé:**
   - ¿El modelo identificó correctamente todas las entidades?
   - ¿Hubo alguna entidad que no detectó? ¿Por qué creés que pasó?
   - ¿Detectó alguna entidad incorrectamente?

Escribí tu CV ficticio en la siguiente celda:

In [13]:
# Pongo algunas cosas de  mi cv en especial a ver como reconoce el instituto de ingles
mi_cv = """Micaela Belen Dominguez es una desarrolladora Full Stack con orientación 
en Data Science, actualmente radicada en Buenos Aires. Estudia Ingeniería 
en Informática en la Universidad Nacional Arturo Jauretche y cursa la Tecnicatura 
en Ciencia de Datos e Inteligencia Artificial. Realizó una pasantía en el Ministerio 
de Educación del Gobierno de la Ciudad de Buenos Aires, donde colaboró en el diseño 
de un sistema de gestión de concursos docentes para la Dirección de Educación Técnico 
Superior. También completó la capacitación de Desarrollador Full Stack Java en 
Talento Tech. Actualmente realiza una capacitación formal en Visma, en el marco del 
programa Educación IT de Fundación Pescar, financiado por JP Morgan. Desarrolló el 
proyecto Huellitas, un e-commerce con frontend en React y backend en .NET. 
Estudió inglés en el Instituto de Cultura Británica."""

# Procesamos
mis_entidades = extractor_ner(mi_cv)

# Mostramos
df_mis_entidades = pd.DataFrame(mis_entidades)
print("Entidades detectadas:")
print(df_mis_entidades[['entity_group', 'word', 'score']].round(3))

# Organizamos
mi_info = {'PER': [], 'ORG': [], 'LOC': []}
for e in mis_entidades:
    if e['entity_group'] in mi_info:
        mi_info[e['entity_group']].append(e['word'])

print("\n" + "="*60)
print(f"CANDIDATO/A: {', '.join(mi_info['PER'])}")
print(f"EMPRESAS/INSTITUCIONES: {', '.join(mi_info['ORG'])}")
print(f"UBICACIONES: {', '.join(mi_info['LOC'])}")

Entidades detectadas:
   entity_group                                               word  score
0           PER                            Micaela Belen Dominguez  0.992
1          MISC                                         Full Stack  0.839
2          MISC                                       Data Science  0.999
3           LOC                                       Buenos Aires  0.999
4          MISC                          Ingeniería en Informática  0.998
5           ORG              Universidad Nacional Arturo Jauretche  0.999
6          MISC                                               Tecn  1.000
7          MISC                                          ##icatura  0.797
8          MISC                                   Ciencia de Datos  0.996
9          MISC                            Inteligencia Artificial  0.886
10          ORG  Ministerio de Educación del Gobierno de la Ciu...  1.000
11          ORG            Dirección de Educación Técnico Superior  1.000
12         MISC 

el modelo maneja bien nombres institucionales largos y formales como ministerios, universidades, fundaciones pero falla con nombres de empresas tech modernas y cortas como Visma, Talento Tech, React que no estaban en su corpus de entrenamiento. Justamente las empresas más relevantes para tu perfil son las que peor detecta eso seria un problema real porque descartaria o tendria resultados malos a la hora de usar esa informacion para una busqueda laboral

---

## Ejercicio 3: Chatbot de Soporte Técnico Automático

### Contexto del problema

Trabajás en una empresa que vende electrodomésticos online. Los clientes suelen hacer preguntas frecuentes sobre garantías, envíos y devoluciones. Querés automatizar las respuestas a estas consultas usando un sistema de Question Answering (QA) que pueda responder basándose en la información de tus políticas.

### Aplicación real

Este tipo de sistemas se usan en:
- Chatbots de atención al cliente (WhatsApp, web)
- Sistemas de FAQ automáticas
- Asistentes virtuales corporativos

### ¿Qué vamos a hacer?

1. Cargar un modelo de Question Answering en español
2. Definir un contexto (políticas de la empresa)
3. Hacer preguntas sobre ese contexto
4. Generar respuestas automáticas

---

### Paso 1: Cargar los modelos necesarios

In [14]:
# Modelo de Question Answering (responder preguntas basadas en contexto)
qa_modelo = pipeline(
    "question-answering",
    model="mrm8488/bert-multi-cased-finetuned-xquadv1"
)

print("Modelo de QA cargado correctamente.")

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

c:\dominguez-micaela-belen-pln-1c-2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\domin\.cache\huggingface\hub\models--mrm8488--bert-multi-cased-finetuned-xquadv1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of the model checkpoint at mrm8488/bert-multi-cased-finetuned-xquadv

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Modelo de QA cargado correctamente.


c:\dominguez-micaela-belen-pln-1c-2026\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Paso 2: Definir el contexto (políticas de la empresa)

Este es el documento que el chatbot va a usar para responder preguntas:

In [15]:
politicas_empresa = """POLÍTICAS DE GARANTÍA Y DEVOLUCIONES - ELECTROHOGAR S.A.

GARANTÍA: Todos nuestros productos tienen garantía oficial de 12 meses contra defectos
de fábrica. La garantía comienza a contar desde la fecha de recepción del producto.
Para hacer válida la garantía, el cliente debe presentar la factura de compra original
y el producto no debe tener daños físicos causados por mal uso.

ENVÍOS: Realizamos envíos a todo el país. En Capital Federal y Gran Buenos Aires,
el envío demora entre 24 y 48 horas hábiles. Para el interior del país, el tiempo
de entrega es de 3 a 7 días hábiles. El envío es gratuito para compras superiores
a $50.000. Para montos menores, se cobra un adicional de $2.500.

DEVOLUCIONES: El cliente tiene 10 días corridos desde la recepción del producto para
solicitar una devolución si el artículo no cumple con sus expectativas. El producto
debe estar sin usar, en su empaque original y con todos sus accesorios. El cliente
debe hacerse cargo del costo de envío de devolución. Una vez recibido y verificado
el producto, reintegramos el 100% del dinero en un plazo de 15 días hábiles.
"""

print("Contexto cargado:")
print(politicas_empresa)

Contexto cargado:
POLÍTICAS DE GARANTÍA Y DEVOLUCIONES - ELECTROHOGAR S.A.

GARANTÍA: Todos nuestros productos tienen garantía oficial de 12 meses contra defectos
de fábrica. La garantía comienza a contar desde la fecha de recepción del producto.
Para hacer válida la garantía, el cliente debe presentar la factura de compra original
y el producto no debe tener daños físicos causados por mal uso.

ENVÍOS: Realizamos envíos a todo el país. En Capital Federal y Gran Buenos Aires,
el envío demora entre 24 y 48 horas hábiles. Para el interior del país, el tiempo
de entrega es de 3 a 7 días hábiles. El envío es gratuito para compras superiores
a $50.000. Para montos menores, se cobra un adicional de $2.500.

DEVOLUCIONES: El cliente tiene 10 días corridos desde la recepción del producto para
solicitar una devolución si el artículo no cumple con sus expectativas. El producto
debe estar sin usar, en su empaque original y con todos sus accesorios. El cliente
debe hacerse cargo del costo de envío

### Paso 3: Hacer preguntas al sistema

In [16]:
# Definimos preguntas típicas de clientes
preguntas = [
    "¿Cuánto dura la garantía?",
    "¿Cuánto tarda el envío a Capital Federal?",
    "¿Cuántos días tengo para devolver un producto?",
    "¿Qué necesito para hacer válida la garantía?",
    "¿El envío es gratis?"
]

# Procesamos cada pregunta
print("RESPUESTAS AUTOMÁTICAS DEL CHATBOT")
print("="*60)

for pregunta in preguntas:
    respuesta = qa_modelo(question=pregunta, context=politicas_empresa)
    print(f"\nPREGUNTA: {pregunta}")
    print(f"RESPUESTA: {respuesta['answer']}")
    print(f"Confianza: {respuesta['score']*100:.1f}%")
    print("-"*60)

RESPUESTAS AUTOMÁTICAS DEL CHATBOT

PREGUNTA: ¿Cuánto dura la garantía?
RESPUESTA: la fecha de recepción del producto
Confianza: 0.3%
------------------------------------------------------------

PREGUNTA: ¿Cuánto tarda el envío a Capital Federal?
RESPUESTA: entre 24 y 48 horas hábiles
Confianza: 55.7%
------------------------------------------------------------

PREGUNTA: ¿Cuántos días tengo para devolver un producto?
RESPUESTA: 10
Confianza: 0.0%
------------------------------------------------------------

PREGUNTA: ¿Qué necesito para hacer válida la garantía?
RESPUESTA: el cliente debe presentar la factura de compra original
Confianza: 65.2%
------------------------------------------------------------

PREGUNTA: ¿El envío es gratis?
RESPUESTA: para compras superiores
a $50.000
Confianza: 78.3%
------------------------------------------------------------


### Paso 4: Crear una función interactiva de chatbot

Vamos a crear una función que simule un chatbot completo:

In [17]:
def chatbot_soporte(pregunta, contexto=politicas_empresa):
    """
    Función que simula un chatbot de soporte técnico.

    Args:
        pregunta (str): La pregunta del cliente
        contexto (str): El documento con las políticas de la empresa

    Returns:
        str: Respuesta formateada para el cliente
    """
    resultado = qa_modelo(question=pregunta, context=contexto)

    # Formateamos la respuesta de manera amigable
    confianza = resultado['score']

    if confianza > 0.5:
        respuesta = f"""Hola! Te respondo tu consulta:

{resultado['answer']}

¿Te fue útil esta información? Si necesitás más detalles, no dudes en consultarnos."""
    else:
        respuesta = """Hola! No encontré una respuesta clara a tu consulta en nuestras
políticas. Te recomiendo que te comuniques con nuestro equipo de atención al cliente
al 0800-XXX-XXXX para que puedan ayudarte mejor."""

    return respuesta

# Probamos la función
print(chatbot_soporte("¿Puedo devolver un producto después de 2 semanas?"))

Hola! No encontré una respuesta clara a tu consulta en nuestras
políticas. Te recomiendo que te comuniques con nuestro equipo de atención al cliente
al 0800-XXX-XXXX para que puedan ayudarte mejor.


## Consigna de Lectura e Interpretación

Ahora es tu turno. Realizá las siguientes tareas:

1. **Escribí 3 preguntas nuevas** que un cliente podría hacer sobre las políticas
2. **Probá el chatbot** con esas preguntas
3. **Modificá el contexto** (politicas_empresa) agregando información nueva, por ejemplo:
   - Formas de pago aceptadas
   - Horarios de atención
   - Información sobre instalación de productos
4. **Respondé:**
   - ¿Las respuestas fueron precisas?
   - ¿Hubo alguna pregunta que el modelo no pudo responder bien?
   - ¿Qué pasa si hacés una pregunta sobre algo que NO está en el contexto?

Usá las siguientes celdas para experimentar:

In [ ]:
# TUS PREGUNTAS ACÁ
mis_preguntas = [
    "¿Cuanto me cobran de envio si compro algo de 30.000 pesos?",
    "¿Puedo devolver un producto que ya use?",
    "¿Qué pasa si el producto tiene daños por mal uso?"
]

# Probamos el chatbot con tus preguntas
for pregunta in mis_preguntas:
    print("\n" + "="*60)
    print(chatbot_soporte(pregunta))


Hola! No encontré una respuesta clara a tu consulta en nuestras
políticas. Te recomiendo que te comuniques con nuestro equipo de atención al cliente
al 0800-XXX-XXXX para que puedan ayudarte mejor.

Hola! No encontré una respuesta clara a tu consulta en nuestras
políticas. Te recomiendo que te comuniques con nuestro equipo de atención al cliente
al 0800-XXX-XXXX para que puedan ayudarte mejor.

Hola! Te respondo tu consulta:

no debe tener daños físicos

¿Te fue útil esta información? Si necesitás más detalles, no dudes en consultarnos.


In [19]:
# MODIFICÁ EL CONTEXTO ACÁ
mi_contexto_ampliado = """POLÍTICAS DE GARANTÍA Y DEVOLUCIONES - ELECTROHOGAR S.A.

GARANTÍA: Todos nuestros productos tienen garantía oficial de 12 meses contra defectos
de fábrica. La garantía comienza a contar desde la fecha de recepción del producto.
Para hacer válida la garantía, el cliente debe presentar la factura de compra original
y el producto no debe tener daños físicos causados por mal uso.

ENVÍOS: Realizamos envíos a todo el país. En Capital Federal y Gran Buenos Aires,
el envío demora entre 24 y 48 horas hábiles. Para el interior del país, el tiempo
de entrega es de 3 a 7 días hábiles. El envío es gratuito para compras superiores
a $50.000. Para montos menores, se cobra un adicional de $2.500.

DEVOLUCIONES: El cliente tiene 10 días corridos desde la recepción del producto para
solicitar una devolución si el artículo no cumple con sus expectativas. El producto
debe estar sin usar, en su empaque original y con todos sus accesorios.

FORMAS DE PAGO: Aceptamos tarjetas de crédito y débito de todas las redes, 
transferencia bancaria y MercadoPago. Los pagos con tarjeta de crédito pueden 
realizarse en hasta 12 cuotas sin interés.

HORARIOS DE ATENCIÓN: Nuestro equipo de atención al cliente está disponible de 
lunes a viernes de 9 a 18hs y sábados de 9 a 13hs a través del 0800-333-1234 
o por WhatsApp al +54 11 4444-5555.

INSTALACIÓN: Ofrecemos servicio de instalación para aires acondicionados y 
lavarropas con un costo adicional de $5.000. El turno debe coordinarse con 
al menos 48 horas de anticipación.
"""

# Probá con una pregunta sobre la info nueva
print(chatbot_soporte("¿Puedo pagar en cuotas?", contexto=mi_contexto_ampliado))
print("\n" + "="*60)
print(chatbot_soporte("¿Cuál es el horario de atención?", contexto=mi_contexto_ampliado))
print("\n" + "="*60)
print(chatbot_soporte("¿Tienen servicio de instalación?", contexto=mi_contexto_ampliado))

Hola! Te respondo tu consulta:

12

¿Te fue útil esta información? Si necesitás más detalles, no dudes en consultarnos.

Hola! No encontré una respuesta clara a tu consulta en nuestras
políticas. Te recomiendo que te comuniques con nuestro equipo de atención al cliente
al 0800-XXX-XXXX para que puedan ayudarte mejor.

Hola! Te respondo tu consulta:

aires acondicionados y 
lavarropas

¿Te fue útil esta información? Si necesitás más detalles, no dudes en consultarnos.


Bastante bien contesti con el contexto ampliado pero igual no pudo resoinder el horario. El QA solo esta devolviendo fragmentos minimos no respuestas completas.

---

## Ejercicio 4: Desafío Autónomo - Análisis de Reseñas de Restaurantes

### Contexto del problema

Sos el encargado de marketing digital de una cadena de restaurantes porteña. Querés implementar un sistema inteligente que procese automáticamente las reseñas que los clientes dejan en Google Maps y redes sociales para:

1. Identificar si la reseña es positiva, negativa o neutral
2. Extraer información clave: nombres de platos mencionados, ubicaciones de las sucursales, nombres de empleados destacados
3. Responder automáticamente a preguntas frecuentes basándose en el menú y políticas del restaurante

### Aplicación real

Este tipo de sistemas combinados se usan en:
- Gestión de reputación online para cadenas de restaurantes
- Análisis de feedback de clientes en hotelería y turismo
- Sistemas de CRM (Customer Relationship Management) inteligentes
- Plataformas de delivery con análisis de satisfacción del cliente

---

### Tu tarea

**Este ejercicio lo tenés que resolver completamente solo**, aplicando todo lo que aprendiste en los ejercicios anteriores. No hay código de ejemplo, solo las instrucciones.

### Parte 1: Análisis de sentimiento de reseñas (30%)

1. Creá una lista con al menos 5 reseñas ficticias de clientes sobre un restaurante argentino (podés inventarlas o buscar reales)
2. Cargá un modelo de análisis de sentimientos en español (buscá en Hugging Face)
3. Clasificá cada reseña y mostrá los resultados en un DataFrame
4. Identificá cuántas reseñas son positivas, negativas y neutrales (si el modelo lo soporta)

**Pistas:**
- Usá `pipeline("text-classification", model=...)`
- Recordá importar `pandas` para crear el DataFrame
- Modelos sugeridos: `finiteautomata/beto-sentiment-analysis` o `pysentimiento/robertuito-sentiment-analysis`

---

### Parte 2: Extracción de información (40%)

1. Tomá 2 de las reseñas que creaste (las más largas y detalladas)
2. Cargá un modelo de NER en español
3. Extraé todas las entidades nombradas de esas reseñas
4. Organizá la información en categorías (personas, lugares, organizaciones)
5. Bonus: ¿Se mencionan nombres de platos? (Nota: el modelo podría no detectarlos como entidades, reflexioná sobre por qué)

**Pistas:**
- Usá `pipeline("ner", model=..., aggregation_strategy="simple")`
- Modelo sugerido: `mrm8488/bert-spanish-cased-finetuned-ner`
- Recordá iterar sobre los resultados para organizarlos por tipo

---

### Parte 3: Sistema de preguntas y respuestas (30%)

1. Escribí un texto con información del restaurante (menú, horarios, ubicación, políticas de reservas, precios promedio, etc.). Mínimo 4-5 oraciones.
2. Cargá un modelo de Question Answering en español
3. Formulá al menos 4 preguntas que un cliente podría hacer
4. Generá respuestas automáticas usando el modelo
5. Mostrá cada pregunta con su respuesta y el nivel de confianza del modelo

**Pistas:**
- Usá `pipeline("question-answering", model=...)`
- Modelo sugerido: `PlanTL-GOB-ES/roberta-base-bne-sqac`
- La función necesita dos parámetros: `question=` y `context=`

---

### Bonus (opcional): Integración completa

Si terminaste las tres partes, intentá crear una función que:
1. Reciba una reseña de cliente como input
2. Analice el sentimiento
3. Extraiga entidades mencionadas
4. Genere un resumen estructurado

Por ejemplo:
```
RESEÑA: "Fui ayer a la sucursal de Palermo y el mozo Juan me atendió bárbaro..."

ANÁLISIS:
- Sentimiento: POSITIVO (95% confianza)
- Empleado mencionado: Juan
- Sucursal: Palermo
- Recomendación: Enviar agradecimiento personalizado
```

---

### Criterios de evaluación

Evaluá tu propio trabajo considerando:

1. **Funcionalidad (50%):** ¿El código funciona sin errores? ¿Completaste las tres partes?
2. **Calidad de datos (20%):** ¿Las reseñas y preguntas son realistas? ¿El contexto tiene información útil?
3. **Presentación (20%):** ¿Los resultados se muestran de forma clara? ¿Usaste DataFrames o print statements organizados?
4. **Reflexión crítica (10%):** ¿Analizaste la calidad de las predicciones? ¿Identificaste limitaciones?

---

### Espacio para tu solución

Usá las celdas siguientes para resolver el desafío. Podés crear todas las celdas que necesites.


In [20]:
# PARTE 1: ANÁLISIS DE SENTIMIENTO
import pandas as pd

reseñas = [
    "Fui a la sucursal de Palermo y quedé encantada. La milanesa napolitana era enorme y estaba perfecta. El mozo Juan nos atendió con mucha amabilidad, volvería sin dudarlo.",
    "Pésima experiencia. Esperamos más de una hora para que nos tomaran el pedido y la carne llegó fría. Nunca más vuelvo a este lugar.",
    "El ambiente está muy bien decorado y la música era agradable. La comida estuvo bien, nada extraordinario. El precio me pareció un poco alto para lo que ofrecen.",
    "Re copado el lugar, los mozos re atentos y la porción de papas fritas era gigante. La hamburguesa con cheddar es un must, la recomiendo.",
    "Fui con mi familia el domingo y fue un desastre total. Se equivocaron con tres pedidos, tardaron dos horas y encima el postre llegó antes que el plato principal.",
    "Zafa. La comida llegó a tiempo y estaba caliente, pero nada que me haya sorprendido. El local de San Telmo tiene mejor onda que este.",
    "El chef Pablo salió a saludarnos personalmente, eso me pareció un detalle increíble. El risotto de hongos era sublime. Una experiencia de diez puntos."
]

resultados = clasificador(reseñas)

df_reseñas = pd.DataFrame({
    'Reseña': reseñas,
    'Sentimiento': [r['label'] for r in resultados],
    'Confianza': [round(r['score'], 3) for r in resultados]
})

print(df_reseñas.to_string())
print("\n" + "="*60)
print("RESUMEN:")
print(df_reseñas['Sentimiento'].value_counts().to_string())

                                                                                                                                                                      Reseña Sentimiento  Confianza
0  Fui a la sucursal de Palermo y quedé encantada. La milanesa napolitana era enorme y estaba perfecta. El mozo Juan nos atendió con mucha amabilidad, volvería sin dudarlo.         POS      0.999
1                                         Pésima experiencia. Esperamos más de una hora para que nos tomaran el pedido y la carne llegó fría. Nunca más vuelvo a este lugar.         NEG      0.999
2           El ambiente está muy bien decorado y la música era agradable. La comida estuvo bien, nada extraordinario. El precio me pareció un poco alto para lo que ofrecen.         POS      0.990
3                                   Re copado el lugar, los mozos re atentos y la porción de papas fritas era gigante. La hamburguesa con cheddar es un must, la recomiendo.         POS      0.998
4          Fui con m

In [22]:
# PARTE 2: EXTRACCIÓN DE INFORMACIÓN (NER)

reseñas_largas = [
    "Fui a la sucursal de Palermo y quedé encantada. La milanesa napolitana era enorme y estaba perfecta. El mozo Juan nos atendió con mucha amabilidad, volvería sin dudarlo.",
    "El chef Pablo salió a saludarnos personalmente, eso me pareció un detalle increíble. El risotto de hongos era sublime. Una experiencia de diez puntos."
]

print("EXTRACCIÓN DE ENTIDADES POR RESEÑA")
print("="*60)

for i, reseña in enumerate(reseñas_largas, 1):
    print(f"\nRESEÑA {i}: '{reseña[:60]}...'")
    entidades = extractor_ner(reseña)
    
    info = {'PER': [], 'ORG': [], 'LOC': [], 'MISC': []}
    for e in entidades:
        tipo = e['entity_group']
        if tipo in info:
            info[tipo].append(e['word'])
    
    print(f"  Personas mencionadas: {', '.join(info['PER']) or 'ninguna'}")
    print(f"  Organizaciones:       {', '.join(info['ORG']) or 'ninguna'}")
    print(f"  Ubicaciones:          {', '.join(info['LOC']) or 'ninguna'}")
    print(f"  Misceláneos (MISC):   {', '.join(info['MISC']) or 'ninguna'}")




EXTRACCIÓN DE ENTIDADES POR RESEÑA

RESEÑA 1: 'Fui a la sucursal de Palermo y quedé encantada. La milanesa ...'
  Personas mencionadas: Juan
  Organizaciones:       ninguna
  Ubicaciones:          Palermo
  Misceláneos (MISC):   ninguna

RESEÑA 2: 'El chef Pablo salió a saludarnos personalmente, eso me parec...'
  Personas mencionadas: Pablo
  Organizaciones:       ninguna
  Ubicaciones:          ninguna
  Misceláneos (MISC):   ninguna


In [23]:
# PARTE 3: QUESTION ANSWERING

info_restaurante = """LA ESQUINA PORTEÑA - Información General

UBICACIÓN Y SUCURSALES: Contamos con tres sucursales en Buenos Aires: 
Palermo (Av. Santa Fe 3500), San Telmo (Defensa 1200) y Belgrano (Cabildo 2100).

HORARIOS: Abrimos de martes a domingo de 12:00 a 16:00hs y de 20:00 a 24:00hs. 
Los lunes permanecemos cerrados.

MENÚ Y PRECIOS: Nuestro plato estrella es la milanesa napolitana a $8.500. 
La hamburguesa con cheddar cuesta $7.200. El risotto de hongos vale $9.000. 
Los postres van desde $3.000 hasta $4.500.

RESERVAS: Aceptamos reservas con hasta 48 horas de anticipación llamando al 
011-4444-5678 o por Instagram (@laesquinaporteña). Para grupos de más de 8 personas 
es obligatorio reservar con anticipación.

MEDIOS DE PAGO: Aceptamos efectivo, tarjetas de débito y crédito, y MercadoPago. 
No cobramos adicional por uso de tarjeta.
"""

preguntas_clientes = [
    "¿Dónde están ubicados?",
    "¿Cuáles son los horarios?",
    "¿Cuánto cuesta la milanesa?",
    "¿Cómo puedo hacer una reserva?",
]

print("CHATBOT - LA ESQUINA PORTEÑA")
print("="*60)
for pregunta in preguntas_clientes:
    respuesta = qa_modelo(question=pregunta, context=info_restaurante)
    print(f"\nPREGUNTA: {pregunta}")
    print(f"RESPUESTA: {respuesta['answer']}")
    print(f"Confianza: {respuesta['score']*100:.1f}%")
    print("-"*60)



CHATBOT - LA ESQUINA PORTEÑA

PREGUNTA: ¿Dónde están ubicados?
RESPUESTA: Cabildo 2100
Confianza: 0.9%
------------------------------------------------------------

PREGUNTA: ¿Cuáles son los horarios?
RESPUESTA: de 20:00 a 24:00hs
Confianza: 33.7%
------------------------------------------------------------

PREGUNTA: ¿Cuánto cuesta la milanesa?
RESPUESTA: $8.500
Confianza: 4.1%
------------------------------------------------------------

PREGUNTA: ¿Cómo puedo hacer una reserva?
RESPUESTA: anticipación
Confianza: 0.2%
------------------------------------------------------------


In [25]:
# BONUS: INTEGRACIÓN COMPLETA
def analizar_reseña_completa(reseña):
    sent = clasificador(reseña)[0]
    etiquetas = {"POS": "POSITIVO", "NEG": "NEGATIVO", "NEU": "NEUTRAL"}
    sentimiento = etiquetas[sent['label']]
    confianza = round(sent['score'] * 100, 1)
    
    entidades = extractor_ner(reseña)
    personas = [e['word'] for e in entidades if e['entity_group'] == 'PER']
    lugares  = [e['word'] for e in entidades if e['entity_group'] == 'LOC']
    
    if sent['label'] == 'POS':
        accion = "Agradecer públicamente y destacar en redes sociales."
    elif sent['label'] == 'NEG':
        accion = "Contactar al cliente para resolver el problema de forma prioritaria."
    else:
        accion = "Monitorear y responder con información adicional."
    
    print(f"RESEÑA: '{reseña[:80]}...'")
    print(f"\nANÁLISIS:")
    print(f"  Sentimiento:            {sentimiento} ({confianza}% confianza)")
    print(f"  Empleados mencionados:  {', '.join(personas) or 'ninguno'}")
    print(f"  Sucursal/Ubicación:     {', '.join(lugares) or 'no detectada'}")
    print(f"  Acción recomendada:     {accion}")

analizar_reseña_completa(reseñas[0])
print("\n" + "="*60)
analizar_reseña_completa(reseñas[1])


RESEÑA: 'Fui a la sucursal de Palermo y quedé encantada. La milanesa napolitana era enorm...'

ANÁLISIS:
  Sentimiento:            POSITIVO (99.9% confianza)
  Empleados mencionados:  Juan
  Sucursal/Ubicación:     Palermo
  Acción recomendada:     Agradecer públicamente y destacar en redes sociales.

RESEÑA: 'Pésima experiencia. Esperamos más de una hora para que nos tomaran el pedido y l...'

ANÁLISIS:
  Sentimiento:            NEGATIVO (99.9% confianza)
  Empleados mencionados:  ninguno
  Sucursal/Ubicación:     no detectada
  Acción recomendada:     Contactar al cliente para resolver el problema de forma prioritaria.


### Reflexión personal

Después de completar el ejercicio, respondé estas preguntas:

1. **¿Qué fue lo más difícil del ejercicio?**
   No me corria el modelo QA `PlanTL-GOB-ES/roberta-base-bne-sqac` e investigando supuestamente ya no esta disponible en HF tuve que cambiarlo.

2. **¿Encontraste alguna limitación en los modelos?**
   El modelo de sentimiento clasifico como neutral "zafa" y aca en argentina tiene como una conotacion levemente positiva. En el caso del NER  no detecta los nombres de los platos como entidades  eso seria una limitacion.

3. **¿Cómo podrías mejorar este sistema para un caso real?**
   Categorizar lo que necesito por ejemplo aca pondria la categoria Comidas.Eso permitiria que el bonus de integracion muestre que plato especifico
   elogio o critico  el cliente, haciendo la accion recomendada util.

4. **¿Qué otras aplicaciones se te ocurren para estas técnicas en el contexto argentino?**
   Creo que se podria aplicar para un sistema que automatice la busqueda labora usando NER para extraer los requisitos 
   de cada oferta puesto comparar con mi CV y generar automáticamente una versión adaptada 
   al puesto. Combinado con una base de datos de contactos de RRHH, podria enviar el CV personalizado por mail sin tener que hacerlo de forma manual para cada empresa. Perdi un monton de tiempo solo para recibir ghosting asi que automatizaria eso. Ademas considero, que buscar trabajo es un trabajo mas del que nadie habla. 

---

---

## Reflexiones Finales del Curso

Completaste los cuatro ejercicios de NLP con Transformers, incluyendo un desafío autónomo. Vamos a reflexionar sobre lo aprendido:

### Ejercicio 1 - Moderación de Comentarios
- **Aprendiste:** A clasificar texto usando modelos de análisis de sentimientos
- **Aplicación:** Moderación automática de redes sociales, priorización de tickets de soporte
- **Limitaciones:** Los modelos pueden tener problemas con ironía, sarcasmo o lenguaje muy coloquial

### Ejercicio 2 - Extracción de CVs
- **Aprendiste:** A extraer entidades nombradas (personas, organizaciones, lugares) de texto
- **Aplicación:** Automatización de RRHH, análisis de documentos, extracción de información estructurada
- **Limitaciones:** Algunos modelos pueden confundir entidades o no detectar nombres poco comunes

### Ejercicio 3 - Chatbot de Soporte
- **Aprendiste:** A usar Question Answering para responder preguntas basadas en un contexto
- **Aplicación:** Chatbots, asistentes virtuales, sistemas de FAQ automáticas
- **Limitaciones:** El modelo solo puede responder sobre información presente en el contexto

### Ejercicio 4 - Desafío Autónomo
- **Aprendiste:** A combinar múltiples técnicas de NLP para resolver un problema real completo
- **Aplicación:** Sistemas integrados de análisis de feedback, gestión de reputación online
- **Habilidad clave:** Autonomía para investigar, implementar y evaluar soluciones de NLP

### Próximos pasos sugeridos

1. **Explorá más modelos** en [Hugging Face Hub](https://huggingface.co/models)
2. **Combiná técnicas:** Por ejemplo, usá clasificación de sentimientos + QA para un chatbot más inteligente
3. **Experimentá con otros idiomas** o dialectos regionales
4. **Investigá fine-tuning:** Aprendé a ajustar modelos con tus propios datos
5. **Desarrollá un proyecto propio:** Elegí un problema real que te interese y aplicá estas técnicas

---

## Glosario Técnico

### Conceptos fundamentales

**Transformer**  
Arquitectura de red neuronal basada en mecanismos de atención que revolucionó el NLP en 2017. Permite procesar secuencias de texto completas simultáneamente en lugar de palabra por palabra.

**Pipeline**  
Interfaz de alto nivel en Hugging Face que encapsula todo el proceso de preprocesamiento, inferencia y postprocesamiento de un modelo. Facilita el uso de modelos preentrenados con pocas líneas de código.

**Modelo preentrenado**  
Modelo de machine learning que fue entrenado previamente con grandes cantidades de datos. Puede usarse directamente o ajustarse (fine-tuning) para tareas específicas.

**Tokenización**  
Proceso de dividir texto en unidades más pequeñas (tokens) que el modelo puede procesar. Puede ser a nivel de palabras, subpalabras o caracteres.

### Tareas de NLP

**Text Classification (Clasificación de texto)**  
Tarea de asignar una o más etiquetas a un texto. Incluye análisis de sentimientos, detección de spam, clasificación de temas, etc.

**Sentiment Analysis (Análisis de sentimientos)**  
Subtipo de clasificación que identifica la polaridad emocional de un texto (positivo, negativo, neutral). Se usa en redes sociales, reseñas de productos, atención al cliente.

**Named Entity Recognition - NER (Reconocimiento de entidades nombradas)**  
Tarea de identificar y clasificar nombres propios en texto: personas (PER), organizaciones (ORG), ubicaciones (LOC), fechas, cantidades monetarias, etc.

**Question Answering - QA (Respuesta a preguntas)**  
Tarea de responder preguntas en lenguaje natural basándose en un contexto dado. El modelo extrae la respuesta directamente del texto proporcionado.

**Text Generation (Generación de texto)**  
Tarea de crear texto nuevo de manera coherente a partir de un prompt inicial. Incluye completado de texto, escritura creativa, chatbots conversacionales.

**Summarization (Resumen automático)**  
Tarea de condensar un texto largo en una versión más corta manteniendo la información más importante.

**Translation (Traducción automática)**  
Tarea de traducir texto de un idioma a otro usando modelos de secuencia a secuencia.

### Componentes técnicos

**Embedding (Representación vectorial)**  
Representación numérica de palabras o tokens como vectores en un espacio multidimensional. Palabras con significados similares tienen embeddings cercanos.

**Attention (Atención)**  
Mecanismo que permite al modelo enfocarse en diferentes partes del input al procesar cada elemento. Es el componente clave de la arquitectura Transformer.

**Fine-tuning (Ajuste fino)**  
Proceso de tomar un modelo preentrenado y entrenarlo adicionalmente con datos específicos de tu dominio para mejorar su desempeño en tu tarea particular.

**Inference (Inferencia)**  
Proceso de usar un modelo ya entrenado para hacer predicciones sobre datos nuevos. No implica entrenamiento, solo aplicación del modelo.

**Score / Confidence (Puntuación / Confianza)**  
Valor numérico (generalmente entre 0 y 1) que indica qué tan seguro está el modelo de su predicción. Valores más altos indican mayor confianza.

### Modelos mencionados

**BETO**  
Versión de BERT (Bidirectional Encoder Representations from Transformers) entrenada específicamente con texto en español. Usado en análisis de sentimientos y otras tareas de clasificación.

**RoBERTa**  
Variante optimizada de BERT con mejoras en el proceso de preentrenamiento. Usado para múltiples tareas de NLP en español.

**BERT (Bidirectional Encoder Representations from Transformers)**  
Modelo Transformer que lee texto bidireccionalmente (izquierda a derecha y derecha a izquierda simultáneamente) para comprender mejor el contexto.

**GPT (Generative Pre-trained Transformer)**  
Familia de modelos diseñados específicamente para generación de texto. Leen texto de izquierda a derecha y predicen la siguiente palabra.

### Plataformas y librerías

**Hugging Face**  
Plataforma y empresa que desarrolla herramientas de NLP de código abierto. Su librería Transformers es el estándar de facto para trabajar con modelos de lenguaje.

**Hugging Face Hub**  
Repositorio online con miles de modelos preentrenados, datasets y demos interactivas. Permite compartir y descargar modelos fácilmente.

**PyTorch / TensorFlow**  
Frameworks de deep learning usados como backend por la librería Transformers. PyTorch es más común en investigación, TensorFlow en producción.

### Métricas y evaluación

**Label (Etiqueta)**  
Categoría asignada por el modelo a un texto. En análisis de sentimientos: POS (positivo), NEG (negativo), NEU (neutral).

**Aggregation strategy (Estrategia de agregación)**  
En NER, método para combinar tokens que pertenecen a la misma entidad. Por ejemplo, "Buenos" y "Aires" se agrupan en una sola entidad "Buenos Aires".

**Context (Contexto)**  
En QA, el documento o párrafo que contiene la información necesaria para responder la pregunta. El modelo busca la respuesta dentro de este contexto.

---

## Recursos adicionales

- **Documentación oficial de Transformers:** https://huggingface.co/docs/transformers
- **Modelos en español:** https://huggingface.co/models?language=es
- **Curso gratuito de Hugging Face:** https://huggingface.co/course
- **Comunidad en español:** https://huggingface.co/spaces

---

*Este cuaderno fue diseñado con fines educativos para estudiantes de NLP en Argentina.*